# Structure Metrics

This notebook demonstrates `run_structure_metrics`, which computes fast geometric quality metrics on predicted protein structures. We load a bundled example PDB, compute the longest alpha helix length and radius of gyration, and export the results for downstream filtering.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("structure_metrics")
display_overview("structure_metrics")
display_docs_section("structure_metrics", "Background")

# Structure Metrics

Structure Metrics computes structural quality metrics from PDB files, specifically the longest alpha helix length and radius of gyration. These metrics are used to flag structural artifacts such as unrealistically long helices or disordered/extended conformations in predicted protein structures.

## Background

**What does this tool measure/predict?**
This tool computes two structural properties from PDB files:
1. **Longest alpha helix**: The length (in residues) of the longest contiguous [alpha-helical](https://en.wikipedia.org/wiki/Alpha_helix) segment, as determined by secondary structure assignment.
2. **[Radius of gyration](https://en.wikipedia.org/wiki/Radius_of_gyration)**: A measure of the overall compactness of the protein structure; the root-mean-square distance of all atoms from the structure's center of mass.

**Why is this important?**
Structure prediction tools (ESMFold, AlphaFold2) can produce artifacts, especially for generated or unusual sequences. Two common failure modes are:
- **Unrealistically long alpha helices**: Some structure predictors "default" to helical conformations for uncertain regions, producing single helices of 50+ residues that are unlikely in real proteins.
- **Extended/disordered structures**: Proteins that fail to fold properly show high gyration radii, indicating the predicted structure is not compact.

These metrics provide fast, quantitative filters to catch these artifacts before more expensive downstream analyses.

**Scientific foundation:**
- **Secondary structure assignment** uses the [DSSP](https://en.wikipedia.org/wiki/DSSP_(hydrogen_bond_estimation_algorithm))-like algorithm implemented in Biotite, which assigns helix, sheet, and coil states based on backbone geometry and hydrogen bonding patterns.
- **Radius of gyration** is a standard biophysical metric used in [small-angle X-ray scattering](https://en.wikipedia.org/wiki/Small-angle_X-ray_scattering) (SAXS) and polymer physics. For globular proteins, it scales approximately as R_g ∝ N^(0.4) where N is the number of residues.

## Available tools

In [2]:
display_available_tools("structure_metrics")

- **`run_structure_metrics()`** — Compute structural quality metrics (SS percentages, longest helix, gyration radius) from PDB files

### `run_structure_metrics`

`run_structure_metrics` reads each PDB file with Biotite, assigns secondary structure elements via the DSSP-like `annotate_sse()` algorithm, and computes both the longest contiguous alpha helix (in residues) and the radius of gyration (in Angstroms). These two metrics serve as cheap filters to flag common structure-prediction artifacts — runaway single helices, and extended/disordered predictions — before running more expensive downstream analyses.

In [3]:
from pathlib import Path

from proto_tools import StructureMetricsConfig, StructureMetricsInput, StructureMetricsOutput, run_structure_metrics
from proto_tools.tools.structure_scoring.structure_metrics.structure_metrics import example_input

In [4]:
# Display input docs
display_api_reference("structure_metrics", "input", "run_structure_metrics")

# Use the tool's canonical example input — a bundled ESMFold-predicted PDB.
inputs = example_input()

**Input** — `StructureMetricsInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `structures` | `list[proto_tools.entities.structures.structure.Structure]` | required | Structures to compute structure metrics for |

In [5]:
# Display config docs
display_api_reference("structure_metrics", "config", "run_structure_metrics")

# StructureMetricsConfig has no tool-specific parameters; all defaults are used
config = StructureMetricsConfig()

**Config** — `StructureMetricsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cpu'` | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int | None` | `None` | Random seed for reproducible results. |

In [6]:
# Run the structure metrics computation
result = run_structure_metrics(inputs, config)

In [7]:
# Display output docs and inspect results
display_api_reference("structure_metrics", "output", "run_structure_metrics")

m = result.metrics[0]
print(f"Longest alpha helix: {m.longest_alpha_helix} residues")
print(f"Radius of gyration:  {m.gyration_radius:.2f} A")

**Output** — `StructureMetricsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `metrics` | `list[proto_tools.tools.structure_scoring.structure_metrics.structure_metrics.StructureQualityMetrics]` | `[]` | Per-structure quality metrics |

Longest alpha helix: 10 residues
Radius of gyration:  11.75 A


## Filter a Batch of Structures

A common use case is filtering a batch of predicted structures to remove artifacts. The example below applies thresholds calibrated for large globular proteins (~1000–1400 residues) — real helices rarely exceed 50 residues, and well-folded structures of this size typically have a radius of gyration under 45 A.

In [8]:
LONGEST_ALPHA_THRESHOLD = 50
GYRATION_RADIUS_THRESHOLD = 45.0

# Re-score the same Structure a few times to simulate a batch
structure = inputs.structures[0]
batch_inputs = StructureMetricsInput(structures=[structure] * 3)
batch_result = run_structure_metrics(batch_inputs, StructureMetricsConfig())

for i, m in enumerate(batch_result.metrics):
    reasons = []
    if m.longest_alpha_helix >= LONGEST_ALPHA_THRESHOLD:
        reasons.append(f"long helix ({m.longest_alpha_helix} residues)")
    if m.gyration_radius >= GYRATION_RADIUS_THRESHOLD:
        reasons.append(f"high Rg ({m.gyration_radius:.1f} A)")
    status = "FILTERED: " + ", ".join(reasons) if reasons else "PASS"
    print(f"structure[{i}]: {status}")

structure[0]: PASS
structure[1]: PASS
structure[2]: PASS


## Export Results

Structure metrics can be exported to CSV or JSON for downstream analysis or joining with other per-structure annotations.

In [9]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export("structure_metrics", export_path=output_dir, file_format="csv")
print(f"Exported to {output_dir / 'structure_metrics.csv'}")

Exported to example_output/structure_metrics.csv
